# Analysis: BitNet vs Ollama Benchmarks

Visualizations and comparison of speed and quality results.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from configs.benchmark_config import SPEED_CSV, QUALITY_CSV

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv(SPEED_CSV)
print(f"Loaded {len(df)} rows")
df.head()

## 1. Decode tok/s by Model (Bar Chart)

In [ ]:
avg = df.groupby(["model_name", "runner"])["decode_toks"].mean().reset_index()
avg = avg.sort_values("decode_toks", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = avg["runner"].map({"bitnet.cpp": "#e74c3c", "ollama": "#3498db"})
ax.barh(avg["model_name"], avg["decode_toks"], color=colors)
ax.set_xlabel("Decode tok/s (mean across all configs)")
ax.set_title("Decode Speed by Model")
plt.tight_layout()
plt.show()

## 2. Decode tok/s per Prompt Config

In [ ]:
df["config"] = df["prompt_tokens"].astype(str) + "p/" + df["gen_tokens"].astype(str) + "g"

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=df, x="config", y="decode_toks", hue="model_name", ax=ax)
ax.set_ylabel("Decode tok/s")
ax.set_title("Decode Speed by Prompt Configuration")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. tok/s vs Thread Count (BitNet only)

In [ ]:
df_threads = df[df["threads"] != "default"].copy()
if not df_threads.empty:
    df_threads["threads"] = df_threads["threads"].astype(int)
    thread_avg = df_threads.groupby(["model_name", "threads"])["decode_toks"].mean().reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    for model in thread_avg["model_name"].unique():
        subset = thread_avg[thread_avg["model_name"] == model]
        ax.plot(subset["threads"], subset["decode_toks"], marker="o", label=model)
    ax.set_xlabel("Thread Count")
    ax.set_ylabel("Decode tok/s")
    ax.set_title("Decode Speed vs Thread Count")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No thread-varied data available (BitNet benchmarks not run yet)")

## 4. Speed vs RAM Scatter

In [ ]:
scatter = df.groupby("model_name").agg(
    decode_toks=("decode_toks", "mean"),
    peak_ram_mb=("peak_ram_mb", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scatter["peak_ram_mb"], scatter["decode_toks"], s=120, zorder=5)
for _, row in scatter.iterrows():
    ax.annotate(row["model_name"], (row["peak_ram_mb"], row["decode_toks"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xlabel("Peak RAM (MB)")
ax.set_ylabel("Decode tok/s")
ax.set_title("Speed vs Memory (upper-left = best)")
plt.tight_layout()
plt.show()

## 5. Prefill / Decode Heatmap

In [ ]:
if "config" not in df.columns:
    df["config"] = df["prompt_tokens"].astype(str) + "p/" + df["gen_tokens"].astype(str) + "g"

pivot = df.pivot_table(values="decode_toks", index="model_name", columns="config", aggfunc="mean")

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_title("Decode tok/s Heatmap (model × config)")
plt.tight_layout()
plt.show()

## 6. Quality Results (if available)

In [ ]:
if os.path.exists(QUALITY_CSV):
    df_qual = pd.read_csv(QUALITY_CSV)
    print(f"Quality results: {len(df_qual)} rows")
    display(df_qual)

    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    df_qual_melted = df_qual.melt(id_vars=["model_name"], var_name="benchmark", value_name="score")
    sns.barplot(data=df_qual_melted, x="benchmark", y="score", hue="model_name", ax=ax)
    ax.set_title("Quality Benchmarks by Model")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No quality results yet — run lm-eval benchmarks first.")